In [ ]:
import pandas as pd
import numpy as np
import ast
import re

In [4]:
# since df_results is too big, put locally in your current folder (since its not on github)
csv_path = "df_results.csv"
df_raw = pd.read_csv(csv_path)
print(df_raw.shape)
print(df_raw.columns.tolist())

# if you wanna see the shape, but loading all is too long
# df = pd.read_csv("df_results.csv", nrows=1)
# print(df.head())
# print(df.info())

/var/folders/8z/zcsmsmvn04b4vm1nxr6_4v480000gq/T/ipykernel_95681/3903895202.py:3: DtypeWarning: Columns (0: postal_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(csv_path)


(227932, 224)
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'postal_code', 'latitude', 'longitude', 'mrt_nearest', 'mrt_nearby', 'bus_nearest', 'bus_nearby', 'store_nearest', 'store_nearby', 'food_nearest', 'food_nearby', 'health_nearest', 'health_nearby', 'restaurant_nearest', 'restaurant_nearby', 'hospital_nearest', 'hospital_nearby', 'lodging_nearest', 'lodging_nearby', 'finance_nearest', 'finance_nearby', 'cafe_nearest', 'cafe_nearby', 'convenience_store_nearest', 'convenience_store_nearby', 'clothing_store_nearest', 'clothing_store_nearby', 'atm_nearest', 'atm_nearby', 'shopping_mall_nearest', 'shopping_mall_nearby', 'grocery_or_supermarket_nearest', 'grocery_or_supermarket_nearby', 'home_goods_store_nearest', 'home_goods_store_nearby', 'school_nearest', 'school_nearby', 'bakery_nearest', 'bakery_nearby', 'beauty_salon_nearest', 'beauty_salon_nearby', 'transit_station

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

df_raw.head()

month        town flat_type block        street_name storey_range  \
0  2017-01  ANG MO KIO    2 ROOM   406  ANG MO KIO AVE 10     10 TO 12   
1  2017-01  ANG MO KIO    3 ROOM   108   ANG MO KIO AVE 4     01 TO 03   
2  2017-01  ANG MO KIO    3 ROOM   602   ANG MO KIO AVE 5     01 TO 03   
3  2017-01  ANG MO KIO    3 ROOM   465  ANG MO KIO AVE 10     04 TO 06   
4  2017-01  ANG MO KIO    3 ROOM   601   ANG MO KIO AVE 5     01 TO 03   

   floor_area_sqm      flat_model  lease_commence_date     remaining_lease  \
0            44.0        Improved                 1979  61 years 04 months   
1            67.0  New Generation                 1978  60 years 07 months   
2            67.0  New Generation                 1980  62 years 05 months   
3            68.0  New Generation                 1980   62 years 01 month   
4            67.0  New Generation                 1980  62 years 05 months   

   resale_price postal_code  latitude   longitude  \
0      232000.0      560406  1.362005  103.853880   
1      250000.0      560108  1.370966  103.838202   
2      262000.0      560602  1.380709  103.835368   
3      265000.0      560465  1.366201  103.857201   
4      265000.0      560601  1.381041  103.835132   

                                                                                               mrt_nearest  \
0     {'name': 'ANG MO KIO MRT STATION', 'stop_id': 'NS16', 'line': 'NS', 'distance_m': 926.8978925114401}   
1    {'name': 'ANG MO KIO MRT STATION', 'stop_id': 'NS16', 'line': 'NS', 'distance_m': 1316.4295661097703}   
2   {'name': 'YIO CHU KANG MRT STATION', 'stop_id': 'NS15', 'line': 'NS', 'distance_m': 1071.182879336597}   
3     {'name': 'ANG MO KIO MRT STATION', 'stop_id': 'NS16', 'line': 'NS', 'distance_m': 880.4238635885278}   
4  {'name': 'YIO CHU KANG MRT STATION', 'stop_id': 'NS15', 'line': 'NS', 'distance_m': 1094.0129912893726}   

                                                                                                  mrt_nearby  \
0     [{'name': 'ANG MO KIO MRT STATION', 'stop_id': 'NS16', 'line': 'NS', 'distance_m': 926.8978925114401}]   
1    [{'name': 'ANG MO KIO MRT STATION', 'stop_id': 'NS16', 'line': 'NS', 'distance_m': 1316.4295661097703}]   
2   [{'name': 'YIO CHU KANG MRT STATION', 'stop_id': 'NS15', 'line': 'NS', 'distance_m': 1071.182879336597}]   
3     [{'name': 'ANG MO KIO MRT STATION', 'stop_id': 'NS16', 'line': 'NS', 'distance_m': 880.4238635885278}]   
4  [{'name': 'YIO CHU KANG MRT STATION', 'stop_id': 'NS15', 'line': 'NS', 'distance_m': 1094.0129912893726}]   

                                                                                  bus_nearest  \
0  {'busstop_code': 54319, 'name': 'Opp Christ The King Ch', 'distance_m': 91.92439852739197}   
1   {'busstop_code': 54189, 'name': 'Mayflower Stn Exit 5', 'distance_m': 166.31237251570192}   
2                 {'busstop_code': 55129, 'name': 'Blk 604', 'distance_m': 129.3446496913248}   
3                 {'busstop_code': 54389, 'name': 'Blk 465', 'distance_m': 69.45283823456104}   
4  {'busstop_code': 55011, 'name': 'Bef Lentor Stn Exit 5', 'distance_m': 149.89790777276488}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

## Data Processing

In [ ]:
# trimmed down the data, its too many columns
# TODO: add distance / time to parent's home & office
base_cols = [
    "month",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "remaining_lease",
    "resale_price",
    "latitude", # replaces postal code
    "longitude",
]

# TODO: we can use poi_nearby (density might be a better indicator)
poi_nearest_cols = [
    "mrt_nearest",
    "bus_nearest",
    "shopping_mall_nearest",
    "grocery_or_supermarket_nearest",
    #"supermarket_nearest", 
    "park_nearest",
    "hospital_nearest", # TODO: can drop
    "doctor_nearest",
    "pharmacy_nearest", 
    "cafe_nearest",
    "restaurant_nearest",
    "gym_nearest",
]

def extract_distance(cell):
    if pd.isna(cell):
        return np.nan
    
    if isinstance(cell, (int, float)):
        return float(cell)
    
    try:
        obj = ast.literal_eval(cell)
        if isinstance(obj, dict):
            return obj.get("distance_m", np.nan)
    except Exception:
        pass
    
    return np.nan

In [13]:
df_flats = df_raw[base_cols + poi_nearest_cols].copy()

for col in poi_nearest_cols:
    new_col = col.replace("_nearest", "_distance_m")
    df_flats[new_col] = df_flats[col].apply(extract_distance)

df_flats = df_flats.drop(columns=poi_nearest_cols)

print(df_flats.shape)
df_flats.head()

(227932, 22)


,month,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,remaining_lease,resale_price,latitude,longitude,mrt_distance_m,bus_distance_m,shopping_mall_distance_m,grocery_or_supermarket_distance_m,park_distance_m,hospital_distance_m,doctor_distance_m,pharmacy_distance_m,cafe_distance_m,restaurant_distance_m,gym_distance_m
0,2017-01,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,61 years 04 months,232000.0,1.362005,103.853880,926.897893,91.924399,1032.440827,298.943356,702.372551,759.478958,759.478958,719.191844,942.041113,675.805030,685.487900
1,2017-01,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,60 years 07 months,250000.0,1.370966,103.838202,1316.429566,166.312373,873.663121,374.856579,445.644348,380.940482,762.518468,1004.302877,1054.606053,259.598745,509.849980
2,2017-01,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,62 years 05 months,262000.0,1.380709,103.835368,1071.182879,129.344650,1531.464976,874.666963,738.579066,602.911934,1704.455313,1734.705553,1711.519153,853.948798,1629.408340
3,2017-01,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,62 years 01 month,265000.0,1.366201,103.857201,880.423864,69.452838,875.833046,574.970520,922.732931,362.911575,881.505494,262.133206,893.065336,176.628039,764.432883
4,2017-01,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,62 years 05 months,265000.0,1.381041,103.835132,1094.012991,149.897908,1575.311125,919.965695,783.742843,606.001012,1749.806066,1778.749292,1754.576301,897.963781,1673.219891


In [14]:
def parse_remaining_lease_to_years(x):
    if pd.isna(x):
        return np.nan
    
    text = str(x).lower()
    years = 0
    months = 0
    
    y_match = re.search(r"(\d+)\s*year", text)
    m_match = re.search(r"(\d+)\s*month", text)
    
    if y_match:
        years = int(y_match.group(1))
    if m_match:
        months = int(m_match.group(1))
    
    return years + months / 12

df_flats["remaining_lease_years"] = df_flats["remaining_lease"].apply(parse_remaining_lease_to_years)

In [15]:
def parse_storey_range_mid(x):
    if pd.isna(x):
        return np.nan
    try:
        parts = str(x).split(" TO ")
        low = int(parts[0])
        high = int(parts[1])
        return (low + high) / 2
    except Exception:
        return np.nan

df_flats["storey_mid"] = df_flats["storey_range"].apply(parse_storey_range_mid)

In [16]:
df_flats["flat_id"] = (
    df_flats["block"].astype(str) + "_" +
    df_flats["street_name"].astype(str) + "_" +
    df_flats["flat_type"].astype(str) + "_" +
    df_flats["month"].astype(str)
)

In [17]:
flat_feature_cols = [
    "flat_id",
    "flat_type",
    "flat_model",
    "floor_area_sqm",
    "resale_price",
    "latitude",
    "longitude",
    "remaining_lease_years",
    "storey_mid",

    # transport
    "mrt_distance_m",
    "bus_distance_m",

    # convenience
    "shopping_mall_distance_m",
    "grocery_or_supermarket_distance_m",

    # lifestyle
    "park_distance_m",
    "cafe_distance_m",
    "restaurant_distance_m",
    "gym_distance_m",

    # health
    "hospital_distance_m",
    "doctor_distance_m",
    "pharmacy_distance_m",
]

df_flats_clean = df_flats[flat_feature_cols].copy()
df_flats_clean.head()

,flat_id,flat_type,flat_model,floor_area_sqm,resale_price,latitude,longitude,remaining_lease_years,storey_mid,mrt_distance_m,bus_distance_m,shopping_mall_distance_m,grocery_or_supermarket_distance_m,park_distance_m,cafe_distance_m,restaurant_distance_m,gym_distance_m,hospital_distance_m,doctor_distance_m,pharmacy_distance_m
0,406_ANG MO KIO AVE 10_2 ROOM_2017-01,2 ROOM,Improved,44.0,232000.0,1.362005,103.853880,61.333333,11.0,926.897893,91.924399,1032.440827,298.943356,702.372551,942.041113,675.805030,685.487900,759.478958,759.478958,719.191844
1,108_ANG MO KIO AVE 4_3 ROOM_2017-01,3 ROOM,New Generation,67.0,250000.0,1.370966,103.838202,60.583333,2.0,1316.429566,166.312373,873.663121,374.856579,445.644348,1054.606053,259.598745,509.849980,380.940482,762.518468,1004.302877
2,602_ANG MO KIO AVE 5_3 ROOM_2017-01,3 ROOM,New Generation,67.0,262000.0,1.380709,103.835368,62.416667,2.0,1071.182879,129.344650,1531.464976,874.666963,738.579066,1711.519153,853.948798,1629.408340,602.911934,1704.455313,1734.705553
3,465_ANG MO KIO AVE 10_3 ROOM_2017-01,3 ROOM,New Generation,68.0,265000.0,1.366201,103.857201,62.083333,5.0,880.423864,69.452838,875.833046,574.970520,922.732931,893.065336,176.628039,764.432883,362.911575,881.505494,262.133206
4,601_ANG MO KIO AVE 5_3 ROOM_2017-01,3 ROOM,New Generation,67.0,265000.0,1.381041,103.835132,62.416667,2.0,1094.012991,149.897908,1575.311125,919.965695,783.742843,1754.576301,897.963781,1673.219891,606.001012,1749.806066,1778.749292


## Represent User Preferences : 

In [ ]:
user_feature_cols = [
    "user_id",
    "budget_max",
    "preferred_flat_type",
    "min_floor_area",
    "preferred_region",
    "max_mrt_distance_m",
    "max_bus_distance_m",
    "weight_budget",
    "weight_transport",
    "weight_food",
    "weight_grocery",
    "weight_health",
    "weight_park",
    "weight_gym",
    "weight_space",
    "weight_lease",
    "weight_storey",
    "weight_location",
]

# df_users = pd.DataFrame(users)
# df_users

## Generate dummy user data

In [21]:
import pandas as pd
import numpy as np

np.random.seed(42)

flat_types = ["2 ROOM", "3 ROOM", "4 ROOM"]
flat_type_probs = [0.45, 0.4, 0.15]

regions = ["North", "North-East", "Central", "East", "West"]
region_probs = [0.18, 0.20, 0.22, 0.20, 0.20]

users = []

for i in range(100):
    users.append({
        "user_id": f"U{i+1}",
        "budget_max": np.random.choice([300000, 350000, 400000, 450000, 500000, 600000]),
        "preferred_flat_type": np.random.choice(flat_types, p=flat_type_probs),
        "min_floor_area": np.random.choice([35, 40, 45, 50, 60]),
        "preferred_region": np.random.choice(regions, p=region_probs),
        "max_mrt_distance_m": np.random.choice([500, 800, 1000, 1200]),
        "max_bus_distance_m": np.random.choice([200, 300, 500]),
        "weight_budget": np.random.choice([3, 4, 5]),
        "weight_transport": np.random.choice([3, 4, 5]),
        "weight_food": np.random.choice([3, 4, 5]),
        "weight_grocery": np.random.choice([3, 4, 5]),
        "weight_health": np.random.choice([1, 2, 3]),
        "weight_park": np.random.choice([1, 2, 3, 4]),
        "weight_gym": np.random.choice([1, 2, 3, 4]),
        "weight_space": np.random.choice([2, 3, 4, 5]),
        "weight_lease": np.random.choice([1, 2, 3]),
        "weight_storey": np.random.choice([1, 2, 3]),
        "weight_location": np.random.choice([2, 3, 4, 5]),
    })

df_users = pd.DataFrame(users)
df_users.head()
df_users = pd.DataFrame(users)

In [23]:
# because we dont expect users to input their preference lat, long
# we just ask them to input preferred region and translate here
region_centers = {
    "North": (1.4360, 103.7860),
    "North-East": (1.3880, 103.8920),
    "Central": (1.3280, 103.8420),
    "East": (1.3570, 103.9400),
    "West": (1.3450, 103.7200),
}

df_users["preferred_lat"] = df_users["preferred_region"].map(lambda r: region_centers[r][0])
df_users["preferred_lon"] = df_users["preferred_region"].map(lambda r: region_centers[r][1])

### Represent data as (user, flat) pairs
We will pair (e.g cross joining) user with the hdbs
Each row is one candidate flat for one user, later the model learns to rank flats within each user group

In [12]:
df_pairs = df_users.merge(df_flats_clean, how="cross")
print(df_pairs.shape)
df_pairs.head()

(22793200, 35)


,user_id,budget_max,preferred_flat_type,weight_budget,weight_transport,weight_school,weight_health,weight_mall,weight_park,weight_food,...,school_distance_m,primary_school_distance_m,secondary_school_distance_m,park_distance_m,hospital_distance_m,doctor_distance_m,pharmacy_distance_m,cafe_distance_m,restaurant_distance_m,gym_distance_m
0,U0,600000,2 ROOM,3,5,5,2,3,3,3,...,285.536120,285.536120,285.536120,702.372551,759.478958,759.478958,719.191844,942.041113,675.805030,685.487900
1,U0,600000,2 ROOM,3,5,5,2,3,3,3,...,422.192741,643.781805,422.192741,445.644348,380.940482,762.518468,1004.302877,1054.606053,259.598745,509.849980
2,U0,600000,2 ROOM,3,5,5,2,3,3,3,...,499.883104,499.883104,762.334388,738.579066,602.911934,1704.455313,1734.705553,1711.519153,853.948798,1629.408340
3,U0,600000,2 ROOM,3,5,5,2,3,3,3,...,501.687201,738.880530,501.687201,922.732931,362.911575,881.505494,262.133206,893.065336,176.628039,764.432883
4,U0,600000,2 ROOM,3,5,5,2,3,3,3,...,539.677677,539.677677,768.747833,783.742843,606.001012,1749.806066,1778.749292,1754.576301,897.963781,1673.219891


In [13]:
def safe_distance_score(dist_m, cap=3000):
    if pd.isna(dist_m):
        return 0.0
    dist_m = min(dist_m, cap)
    return 1 - (dist_m / cap)

df_pairs["budget_fit_score"] = np.where(
    df_pairs["resale_price"] <= df_pairs["budget_max"],
    1 - ((df_pairs["budget_max"] - df_pairs["resale_price"]) / df_pairs["budget_max"]) * 0.2,
    np.maximum(0, 1 - ((df_pairs["resale_price"] - df_pairs["budget_max"]) / df_pairs["budget_max"]))
)

df_pairs["flat_type_match"] = (
    df_pairs["flat_type"].astype(str).str.upper() ==
    df_pairs["preferred_flat_type"].astype(str).str.upper()
).astype(int)

df_pairs["area_fit_score"] = np.where(
    df_pairs["floor_area_sqm"] >= df_pairs["min_floor_area"],
    1.0,
    np.maximum(0, df_pairs["floor_area_sqm"] / df_pairs["min_floor_area"])
)

df_pairs["transport_score"] = (
    df_pairs["mrt_distance_m"].apply(safe_distance_score) * 0.7 +
    df_pairs["bus_distance_m"].apply(safe_distance_score) * 0.3
)

df_pairs["school_score"] = (
    df_pairs["primary_school_distance_m"].apply(safe_distance_score) * 0.5 +
    df_pairs["secondary_school_distance_m"].apply(safe_distance_score) * 0.5
)

df_pairs["health_score"] = (
    df_pairs["hospital_distance_m"].apply(safe_distance_score) * 0.4 +
    df_pairs["doctor_distance_m"].apply(safe_distance_score) * 0.3 +
    df_pairs["pharmacy_distance_m"].apply(safe_distance_score) * 0.3
)

df_pairs["mall_score"] = (
    df_pairs["shopping_mall_distance_m"].apply(safe_distance_score) * 0.5 +
    df_pairs["supermarket_distance_m"].apply(safe_distance_score) * 0.5
)

df_pairs["park_score"] = df_pairs["park_distance_m"].apply(safe_distance_score)

df_pairs["food_score"] = (
    df_pairs["restaurant_distance_m"].apply(safe_distance_score) * 0.5 +
    df_pairs["cafe_distance_m"].apply(safe_distance_score) * 0.5
)

### Synthetic Relevance Label

In [14]:
df_pairs["raw_relevance"] = (
    df_pairs["weight_budget"] * df_pairs["budget_fit_score"] +
    2.0 * df_pairs["flat_type_match"] +
    1.5 * df_pairs["area_fit_score"] +
    df_pairs["weight_transport"] * df_pairs["transport_score"] +
    df_pairs["weight_school"] * df_pairs["school_score"] +
    df_pairs["weight_health"] * df_pairs["health_score"] +
    df_pairs["weight_mall"] * df_pairs["mall_score"] +
    df_pairs["weight_park"] * df_pairs["park_score"] +
    df_pairs["weight_food"] * df_pairs["food_score"]
)

min_score = df_pairs["raw_relevance"].min()
max_score = df_pairs["raw_relevance"].max()

df_pairs["relevance_label"] = (
    4 * (df_pairs["raw_relevance"] - min_score) / (max_score - min_score + 1e-9)
).round().astype(int)

df_pairs[["user_id", "flat_id", "raw_relevance", "relevance_label"]].head()

,user_id,flat_id,raw_relevance,relevance_label
0,U0,406_ANG MO KIO AVE 10_2 ROOM_2017-01,22.086847,3
1,U0,108_ANG MO KIO AVE 4_3 ROOM_2017-01,20.077409,2
2,U0,602_ANG MO KIO AVE 5_3 ROOM_2017-01,18.339445,2
3,U0,465_ANG MO KIO AVE 10_3 ROOM_2017-01,20.194154,2
4,U0,601_ANG MO KIO AVE 5_3 ROOM_2017-01,18.115052,2


## Data Filter
If we directly feed df_pairs, it will throw error because each user is paired with all possible flats : <br>
[LightGBM] [Fatal] Number of rows 227932 exceeds upper limit of 10000 for a query

So we need to first filter for each user the possible flats

### Filter 1 : 

In [15]:
MAX_CANDIDATES_PER_USER = 5000

df_pairs = (
    df_pairs.sort_values(["user_id", "raw_relevance"], ascending=[True, False])
    .groupby("user_id", group_keys=False)
    .head(MAX_CANDIDATES_PER_USER)
    .reset_index(drop=True)
)

print(df_pairs.groupby("user_id").size())
print(df_pairs.shape)


user_id
U0     5000
U1     5000
U10    5000
U11    5000
U12    5000
       ... 
U95    5000
U96    5000
U97    5000
U98    5000
U99    5000
Length: 100, dtype: int64
(500000, 46)


### Filter 2 :

In [16]:
# Other possible filter (but it might not reduce as much as above, to a number)
# Before cross join or before training, remove obviously bad flats:
# way over budget, wrong flat type, too small floor area
def prefilter_user_candidates(user_row, flats_df):
    filtered = flats_df.copy()
    
    filtered = filtered[filtered["resale_price"] <= user_row["budget_max"] * 1.3]
    filtered = filtered[filtered["floor_area_sqm"] >= user_row["min_floor_area"] * 0.8]
    
    # optional: only exact flat type match
    filtered = filtered[filtered["flat_type"] == user_row["preferred_flat_type"]]
    
    return filtered

## Model 1 : LightGBM LambdaMART
### Pros : 
- Best fit for tabular ranking
- Handles numeric/categorical features well
- Fast
- Common for recommendation/ranking problems 

#### Training matrix : 

In [18]:
feature_cols = [
    # user preference features
    "budget_max",
    "weight_budget",
    "weight_transport",
    "weight_school",
    "weight_health",
    "weight_mall",
    "weight_park",
    "weight_food",
    "min_floor_area",

    # flat features
    "floor_area_sqm",
    "resale_price",
    "remaining_lease_years",
    "storey_mid",
    "mrt_distance_m",
    "bus_distance_m",
    "shopping_mall_distance_m",
    "supermarket_distance_m",
    "primary_school_distance_m",
    "secondary_school_distance_m",
    "park_distance_m",
    "hospital_distance_m",
    "doctor_distance_m",
    "pharmacy_distance_m",
    "restaurant_distance_m",
    "cafe_distance_m",
    "gym_distance_m",

    # interaction features
    "budget_fit_score",
    "flat_type_match",
    "area_fit_score",
    "transport_score",
    "school_score",
    "health_score",
    "mall_score",
    "park_score",
    "food_score",
]

#### Train / validation split by user
Ranking split by user groups

In [19]:
user_ids = df_pairs["user_id"].unique().tolist()

train_users = user_ids[: int(len(user_ids) * 0.8)]
val_users = user_ids[int(len(user_ids) * 0.8):]

train_df = df_pairs[df_pairs["user_id"].isin(train_users)].copy()
val_df = df_pairs[df_pairs["user_id"].isin(val_users)].copy()

X_train = train_df[feature_cols]
y_train = train_df["relevance_label"]

X_val = val_df[feature_cols]
y_val = val_df["relevance_label"]

train_group = train_df.groupby("user_id").size().tolist()
val_group = val_df.groupby("user_id").size().tolist()

print(train_df.shape, val_df.shape)
print(train_group, val_group)

(400000, 46) (100000, 46)
[5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000] [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]


#### Training Lambdamart

In [20]:
import lightgbm as lgb

train_data = lgb.Dataset(X_train, label=y_train, group=train_group)
val_data = lgb.Dataset(X_val, label=y_val, group=val_group, reference=train_data)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5, 10],
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 20,
    "verbosity": -1,
}

model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    valid_names=["train", "val"],
    num_boost_round=200,
    callbacks=[
        lgb.early_stopping(20),
        lgb.log_evaluation(20),
    ],
)

Training until validation scores don't improve for 20 rounds
[20]	train's ndcg@5: 1	train's ndcg@10: 0.998373	val's ndcg@5: 0.97737	val's ndcg@10: 0.966445
Early stopping, best iteration is:
[1]	train's ndcg@5: 1	train's ndcg@10: 0.998339	val's ndcg@5: 0.987233	val's ndcg@10: 0.974858


#### Rank flats by each user

In [ ]:
df_pairs["pred_score"] = model.predict(df_pairs[feature_cols])

ranked = df_pairs.sort_values(["user_id", "pred_score"], ascending=[True, False])

ranked[[
    "user_id",
    "flat_id",
    "town",
    "flat_type",
    "resale_price",
    "floor_area_sqm",
    "mrt_distance_m",
    "pred_score",
    "relevance_label"
]].head(20)

# Get top 5 : 
top5 = (
    ranked.groupby("user_id", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5[[
    "user_id",
    "flat_id",
    "town",
    "flat_type",
    "resale_price",
    "floor_area_sqm",
    "mrt_distance_m",
    "pred_score"
]]

# if you wanna show everything and not having the ... shown : 
# print(top5.to_string())

    user_id  budget_max preferred_flat_type  weight_budget  weight_transport  weight_school  weight_health  weight_mall  weight_park  weight_food  min_floor_area                                 flat_id             town flat_type  floor_area_sqm  resale_price  latitude   longitude  remaining_lease_years  storey_mid  mrt_distance_m  bus_distance_m  shopping_mall_distance_m  grocery_or_supermarket_distance_m  supermarket_distance_m  school_distance_m  primary_school_distance_m  secondary_school_distance_m  park_distance_m  hospital_distance_m  doctor_distance_m  pharmacy_distance_m  cafe_distance_m  restaurant_distance_m  gym_distance_m  budget_fit_score  flat_type_match  area_fit_score  transport_score  school_score  health_score  mall_score  park_score  food_score  raw_relevance  relevance_label  pred_score
0        U0      600000              2 ROOM              3                 5              5              2            3            3            3              90      440A_CLEMENTI A

## Model 2 : XGBoost Ranker
XGBoost turns out to be using lambdamart, too as its underlying algorithm. But we can try both

In [23]:
from xgboost import XGBRanker
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

target_col = "relevance_label"
group_col = "user_id"

In [24]:
# Again, split by user
user_ids = df_pairs[group_col].unique().tolist()

train_users, test_users = train_test_split(
    user_ids,
    test_size=0.33,
    random_state=42
)

train_df = df_pairs[df_pairs[group_col].isin(train_users)].copy()
test_df = df_pairs[df_pairs[group_col].isin(test_users)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (335000, 47)
Test shape: (165000, 47)


In [25]:
# Sort before grouping 
train_df = train_df.sort_values([group_col, "flat_id"]).reset_index(drop=True)
test_df = test_df.sort_values([group_col, "flat_id"]).reset_index(drop=True)

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_group = train_df.groupby(group_col).size().tolist()
test_group = test_df.groupby(group_col).size().tolist()

print("Train groups:", train_group)
print("Test groups:", test_group)

Train groups: [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]
Test groups: [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]


### Training XGBoost ranker

In [26]:
xgb_ranker = XGBRanker(
    objective="rank:ndcg",
    learning_rate=0.05,
    max_depth=6,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_ranker.fit(
    X_train,
    y_train,
    group=train_group,
    verbose=True
)

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=0.8, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          learning_rate=0.05, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=200,
          n_jobs=None, num_parallel_tree=None, random_state=42, ...)

### Prediction + rank

In [27]:
test_df["pred_score_xgb"] = xgb_ranker.predict(X_test)

ranked_test_xgb = test_df.sort_values(
    [group_col, "pred_score_xgb"],
    ascending=[True, False]
).reset_index(drop=True)

### Top 5 each user : 

In [28]:
top5_xgb = (
    ranked_test_xgb.groupby(group_col, group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5_xgb[[
    "user_id",
    "flat_id",
    "flat_type",
    "resale_price",
    "floor_area_sqm",
    "mrt_distance_m",
    "pred_score_xgb"
]]

,user_id,flat_id,flat_type,resale_price,floor_area_sqm,mrt_distance_m,pred_score_xgb
0,U0,440A_CLEMENTI AVE 3_2 ROOM_2023-05,2 ROOM,492000.0,47.0,168.726476,4.168982
1,U0,32_NEW MKT RD_2 ROOM_2023-06,2 ROOM,430000.0,63.0,109.391318,3.100995
2,U0,32_NEW MKT RD_2 ROOM_2022-08,2 ROOM,420000.0,63.0,109.391318,3.089757
3,U0,32_NEW MKT RD_2 ROOM_2024-06,2 ROOM,388000.0,52.0,109.391318,2.310727
4,U0,32_NEW MKT RD_2 ROOM_2025-02,2 ROOM,391888.0,52.0,109.391318,2.310727
...,...,...,...,...,...,...,...
160,U96,323_ANG MO KIO AVE 3_2 ROOM_2022-09,2 ROOM,285000.0,44.0,294.003601,4.459447
161,U96,323_ANG MO KIO AVE 3_2 ROOM_2022-10,2 ROOM,280000.0,48.0,294.003601,4.459447
162,U96,323_ANG MO KIO AVE 3_2 ROOM_2023-01,2 ROOM,300000.0,49.0,294.003601,4.459447
163,U96,323_ANG MO KIO AVE 3_2 ROOM_2023-08,2 ROOM,300000.0,44.0,294.003601,4.459447


### NDCG@5 Metrics

In [29]:
def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    if relevances.size == 0:
        return 0.0
    return np.sum((2 ** relevances - 1) / np.log2(np.arange(2, relevances.size + 2)))

def ndcg_at_k(y_true, y_score, k=5):
    order = np.argsort(y_score)[::-1]
    y_true_sorted_by_pred = np.take(y_true, order)
    dcg = dcg_at_k(y_true_sorted_by_pred, k)

    ideal_order = np.argsort(y_true)[::-1]
    y_true_sorted_ideal = np.take(y_true, ideal_order)
    idcg = dcg_at_k(y_true_sorted_ideal, k)

    if idcg == 0:
        return 0.0
    return dcg / idcg

In [30]:
ndcg_scores_xgb = []

for user_id, group in ranked_test_xgb.groupby("user_id"):
    score = ndcg_at_k(
        group["relevance_label"].values,
        group["pred_score_xgb"].values,
        k=5
    )
    ndcg_scores_xgb.append(score)

print("Mean NDCG@5:", np.mean(ndcg_scores_xgb))

Mean NDCG@5: 0.9962946101344671


### Feature importance 

In [31]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb_ranker.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(15)

,feature,importance
31,health_score,0.222464
34,food_score,0.135276
23,restaurant_distance_m,0.067463
27,flat_type_match,0.065089
24,cafe_distance_m,0.059755
33,park_score,0.044962
16,supermarket_distance_m,0.041031
19,park_distance_m,0.034274
20,hospital_distance_m,0.031274
21,doctor_distance_m,0.025450


## Other checks : 

In [32]:
print(df_pairs["relevance_label"].describe())
print(df_pairs["relevance_label"].value_counts())

count    500000.000000
mean          2.493360
std           0.558742
min           1.000000
25%           2.000000
50%           2.000000
75%           3.000000
max           4.000000
Name: relevance_label, dtype: float64
2    253918
3    230523
4     10572
1      4987
Name: relevance_label, dtype: int64


In [33]:
print(X_train.nunique().sort_values())

flat_type_match                   2
budget_max                        5
weight_budget                     5
weight_transport                  5
weight_school                     5
weight_health                     5
weight_mall                       5
weight_park                       5
weight_food                       5
min_floor_area                    5
storey_mid                       17
area_fit_score                  110
floor_area_sqm                  124
remaining_lease_years           679
resale_price                   2263
school_score                   4114
transport_score                4114
mall_score                     4114
cafe_distance_m                4114
gym_distance_m                 4114
health_score                   4114
restaurant_distance_m          4114
primary_school_distance_m      4114
doctor_distance_m              4114
hospital_distance_m            4114
park_distance_m                4114
secondary_school_distance_m    4114
park_score                  

In [34]:
print(train_df.groupby("user_id").size())

user_id
U1     5000
U10    5000
U11    5000
U13    5000
U14    5000
       ... 
U94    5000
U95    5000
U97    5000
U98    5000
U99    5000
Length: 67, dtype: int64


In [35]:
pred = xgb_ranker.predict(X_train)
print(pred[:20])
print(np.unique(pred))

[-4.1829867 -4.0911164 -4.1829867 -4.0911164 -4.1318836 -3.7957642
 -3.8384843 -3.9028563 -3.7957642 -3.9028563 -3.997077  -3.9460073
 -3.9308345 -4.0892186 -3.997077  -3.8247051 -3.9629054 -4.0932636
 -4.0932636 -4.1313744]
[-4.8393483 -4.8074417 -4.8047647 ...  4.4887342  4.4892874  4.4942327]
